In [1]:
!pip3 install -r requirements.txt

  Cloning https://github.com/fastforwardlabs/cmlbootstrap to /tmp/pip-install-vlk0vqzd/cmlbootstrap_983ae199f34f4b0ba9c180276405865e
  Running command git clone --filter=blob:none --quiet https://github.com/fastforwardlabs/cmlbootstrap /tmp/pip-install-vlk0vqzd/cmlbootstrap_983ae199f34f4b0ba9c180276405865e
  Resolved https://github.com/fastforwardlabs/cmlbootstrap to commit dd9c800fd46807fdee9f25d8178f46c5eab06e27
  Preparing metadata (setup.py) ... done


In [3]:
import os, warnings, sys, logging
import mlflow
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score
import mlflow.sklearn
from xgboost import XGBClassifier
from datetime import date
import cml.data_v1 as cmldata
import pyspark.pandas as ps

In [4]:
USERNAME = os.environ["PROJECT_OWNER"]
DBNAME = os.environ["DBNAME_PREFIX"]+"_"+USERNAME
CONNECTION_NAME = os.environ["SPARK_CONNECTION_NAME"]

DATE = date.today()
EXPERIMENT_NAME = "xgb-cc-fraud-{0}".format(USERNAME)

mlflow.set_experiment(EXPERIMENT_NAME)

<Experiment: artifact_location='/home/cdsw/.experiments/ful2-5dof-52y4-f2tt', creation_time=None, experiment_id='ful2-5dof-52y4-f2tt', last_update_time=None, lifecycle_stage='active', name='xgb-cc-fraud-rpakalapati', tags={}>

In [5]:
conn = cmldata.get_connection(CONNECTION_NAME)
spark = conn.get_spark_session()

df_from_sql = ps.read_table('{0}.transactions_{1}'.format(DBNAME, USERNAME))
df = df_from_sql.to_pandas()
#df = df.drop(columns=["job"])

test_size = 0.3
X_train, X_test, y_train, y_test = train_test_split(df.drop("fraud_trx", axis=1), df["fraud_trx"], test_size=test_size)

Setting spark.hadoop.yarn.resourcemanager.principal to rpakalapati


Spark Application Id:spark-application-1774376729615


Hive Session ID = 0936a44f-33d0-4046-88aa-d93ebfc8e525


In [6]:
with mlflow.start_run():

    model = XGBClassifier(use_label_encoder=False, eval_metric="logloss")

    # Step 1: cambiar test_size linea 69 y recorrer
    # Step 2: cambiar linea 74, agregar linea 97, y recorrer
      # linea 75: model = XGBClassifier(use_label_encoder=False, max_depth=4, eval_metric="logloss")
      # linea 97: mlflow.log_param("max_depth", 4)
    # Step 3: cambiar linea 74 y 97, agregar linea 98, y recorrer
      # linea 75: model = XGBClassifier(use_label_encoder=False, max_depth=2, max_leaf_nodes=5, eval_metric="logloss")
      # linea 97: mlflow.log_param("max_depth", 2)
      # linea 98: mlflow.log_param("max_leaf_nodes", 5)

    model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)

    print("Accuracy: %.2f%%" % (accuracy * 100.0))
    print("Test Size: %.2f%%" % (test_size * 100.0))

    mlflow.log_param("accuracy", accuracy)
    mlflow.log_param("test_size", test_size)

    # Step 2:
    # Step 3:

    mlflow.xgboost.log_model(model, artifact_path="artifacts")#, registered_model_name="my_xgboost_model"


/home/cdsw/.local/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [18:28:51] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Accuracy: 90.47%
Test Size: 30.00%


/home/cdsw/.local/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [18:28:51] WARNING: /workspace/src/c_api/c_api.cc:1374: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  warnings.warn(smsg, UserWarning)
2026/03/24 18:29:03 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


In [5]:
def getLatestExperimentInfo(experimentName):
    """
    Method to capture the latest Experiment Id and Run ID for the provided experimentName
    """
    experimentId = mlflow.get_experiment_by_name(experimentName).experiment_id
    runsDf = mlflow.search_runs(experimentId, run_view_type=1)
    experimentId = runsDf.iloc[-1]['experiment_id']
    experimentRunId = runsDf.iloc[-1]['run_id']

    return experimentId, experimentRunId

experimentId, experimentRunId = getLatestExperimentInfo(EXPERIMENT_NAME)

In [6]:
#Replace Experiment Run ID here:
run = mlflow.get_run(experimentRunId)

pd.DataFrame(data=[run.data.params], index=["Value"]).T
pd.DataFrame(data=[run.data.metrics], index=["Value"]).T

client = mlflow.tracking.MlflowClient()
client.list_artifacts(run_id=run.info.run_id)

[<FileInfo: file_size=None, is_dir=True, path='artifacts'>]

In [1]:
!pip3 install nyoka

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.0/304.0 kB 2.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 32.5 MB/s eta 0:00:00:00:0100:01


In [11]:
from sklearn.pipeline import Pipeline
from nyoka import xgboost_to_pmml

# 1. Wrap your trained model in a Pipeline
# (Even if you don't have preprocessing steps, Nyoka needs the Pipeline structure)
pipeline = Pipeline([
    ('classifier', model)
])

# 2. Export using the pipeline object instead of the model
pmml_output_path = "/home/cdsw/pmml/fraud_model_xgboost.pmml"

xgboost_to_pmml(
    pipeline=pipeline, 
    col_names=X_train.columns.tolist(), 
    target_name="fraud_trx", 
    pmml_f_name=pmml_output_path
)

# 3. Log to MLflow
mlflow.log_artifact(pmml_output_path)